In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)
path = os.path.join(path, 'Q1_data.csv')

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=40, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns='Order_ID')

In [ ]:
# Task 2: Write your code here:
df = df.dropna(subset='Delivery_Time')
# there is nothing you can do when the target is missing, just drop. Any other strategy will introduce noise

df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])

df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean()) # i don't think this feature matter that much

df[['Traffic_Level', 'Time_of_Day']] = df[['Traffic_Level', 'Time_of_Day']].fillna('unkown') # i think these features are really important, i don't want to hurt performance by filling them with mean or midean

df.isna().sum()

In [ ]:
# Task 3: Write your code here:
print(df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
ohe = OneHotEncoder()
cat_cols = ['Distance_km',	'Weather',	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type']

label_encoders = {}
for col in cat_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df.head()

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
df[df.columns] = scaler.fit_transform(df[df.columns])

df.head()

In [ ]:
# Task 6: Write your code here:
# this is a regression task, what in the world would be target imbalance?

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns='Delivery_Time').astype(float)
y = df['Delivery_Time'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
folds_error = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  folds_error.append(mae)

print("averaged score across all folds: ", np.mean(folds_error))


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': df.drop(columns='Delivery_Time').columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test)



plt.figure(figsize=(10, 5))
plt.hist(pd.DataFrame(y_pred), bins=40, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: